# 02.03 · Your first agent with `create_agent`

**Module 02 · Tools & Your First Agents** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lesson 02.02 (the loop by hand). |
| 🔑 Keys | `GROQ_API_KEY` (optional), `LANGSMITH_API_KEY` (optional, for traces) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Build** an agent with `create_agent(model, tools, system_prompt=...)` and explain what it adds to the loop you wrote in 02.02.
2. **Read** an agent's graph (`model` and `tools` nodes) and its result (`messages`, the final answer, step counts).
3. **Bound** every run: `recursion_limit`, `GraphRecursionError`, and `ModelCallLimitMiddleware`, and know why the default limit is no protection.
4. **Extend** state with `state_schema` and change the prompt per request with `@dynamic_prompt`.
5. **Choose** the model per step with `wrap_model_call` (cheap model first, bigger one later).
6. **Migrate** old `create_react_agent` code to `create_agent`.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Agents](https://docs.langchain.com/oss/python/langchain/agents) | **Main page.** `create_agent`, the ReAct loop, static and dynamic models and prompts, `state_schema`, invocation |
| [Quickstart](https://docs.langchain.com/oss/python/langchain/quickstart) | A complete agent end to end (tools, prompt, structured output, memory) in one page |
| [`create_agent` reference](https://reference.langchain.com/python/langchain/agents/factory/create_agent) | Every parameter, with types |
| [What's new in LangGraph v1](https://docs.langchain.com/oss/python/releases/langgraph-v1) | Why `create_react_agent` was deprecated |
| [Middleware overview](https://docs.langchain.com/oss/python/langchain/middleware/overview) | The hooks `create_agent` exposes (`before_model`, `wrap_model_call`, ...) |
| [Built-in middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in) | `ModelCallLimitMiddleware`, `ToolCallLimitMiddleware` and friends |
| [LangChain v1 migration guide](https://docs.langchain.com/oss/python/migrate/langchain-v1) | `create_react_agent` → `create_agent`: what was renamed or removed |
| [Graph API → Recursion limit](https://docs.langchain.com/oss/python/langgraph/graph-api#recursion-limit) | What a "super-step" is and how `recursion_limit` counts them |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2"

**You should now see** nothing, or *"Note: you may need to restart the kernel to use updated packages."* Restart only if a package was **upgraded** after you had already imported it (Lesson 00.07). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.03).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-groq", "langgraph",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/02-tools-and-agents/).
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import the course package (installed editable in Lesson 00.02; src/ fallback otherwise).
try:
    import mastery
except ModuleNotFoundError:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    import mastery
    print("ℹ️ Loaded mastery from src/. Fix once: python -m pip install -e \".[dev]\"")
print("Repo root:", REPO_ROOT)

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'LANGSMITH_API_KEY')
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

---
## Part 1 · The loop you wrote, as one function call

`create_agent` builds the loop from 02.02 as a **LangGraph graph** with two nodes:

- **`model`** calls the model with the tools bound (and the system prompt in front),
- **`tools`** runs the requested tool calls (in parallel if there are several) and appends `ToolMessage`s,

and one routing rule: after `model`, go to `tools` if the reply has tool calls, otherwise **end**. This is the **ReAct** pattern (*reason + act*): think, act with a tool, observe the result, repeat.

What you get on top of your hand-written loop: tool errors turned into messages, streaming, persistence (Module 03), middleware hooks, structured output (02.05), tracing per step, and Studio (02.07). Everything later in the course builds on this object.

In [ ]:
# 🧪 Offline: the same two-step task as 02.02, now with create_agent and a scripted model.
from langchain.agents import create_agent
from langchain.messages import AIMessage
from mastery.agents import agent_stats, format_transcript
from mastery.testing import ScriptedChatModel
from mastery.tools import calculator, days_between

SYSTEM_PROMPT = (
    "You are a careful personal-finance assistant. Use the tools for every date or arithmetic step; "
    "never compute in your head. Answer in one or two sentences."
)
QUESTION = "If I save 45,000 COP a day from 2026-10-12 until 2026-12-25, how much will I have?"

scripted = ScriptedChatModel(responses=[
    AIMessage("", tool_calls=[{"name": "days_between", "args": {"start": "2026-10-12", "end": "2026-12-25"}, "id": "c1"}]),
    AIMessage("", tool_calls=[{"name": "calculator", "args": {"expression": "74 * 45000"}, "id": "c2"}]),
    "You'll have 3,330,000 COP after 74 days.",
])
agent = create_agent(scripted, tools=[calculator, days_between], system_prompt=SYSTEM_PROMPT, name="savings_agent")

result = agent.invoke({"messages": [{"role": "user", "content": QUESTION}]})
print(format_transcript(result["messages"]))
print("\nfinal answer:", result["messages"][-1].content)
print("stats       :", agent_stats(result["messages"]))
print("bound tools :", scripted.bound_tools)
print("first message the model saw:", type(scripted.calls[0][0]).__name__, "→", scripted.calls[0][0].content[:50], "...")

**You should now see** the same six-line transcript as in 02.02, `final answer: You'll have 3,330,000 COP after 74 days.`, and stats with 3 model calls and 2 tool calls.

Notice the last line: the model's first message was a **`SystemMessage`** with our prompt. `create_agent` puts it in front of every model call, but it is **not stored** in `result["messages"]`, which starts with your `HumanMessage`. The prompt is configuration, not history.

In [ ]:
# 🧪 Offline: the graph behind the agent. Paste the Mermaid text into https://mermaid.live to see it drawn.
graph = agent.get_graph()
print("nodes:", list(graph.nodes))
print(graph.draw_mermaid())

**You should now see** nodes `['__start__', 'model', 'tools', '__end__']` and dotted (conditional) edges `model → tools`, `model → __end__`, `tools → model`. (Older tutorials call the model node `agent`; LangChain v1 renamed it to `model`.)

In [ ]:
# 💸 Live: the same agent with the real model (3–4 short calls). Traced to LangSmith if tracing is on.
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    live_agent = create_agent(get_model("fast", temperature=0, reasoning_effort="low"),
                              tools=[calculator, days_between], system_prompt=SYSTEM_PROMPT, name="savings_agent")
    live = live_agent.invoke({"messages": [{"role": "user", "content": QUESTION}]},
                             {"recursion_limit": 12, "run_name": "m02-03-first-agent", "tags": ["m02"]})
    print(format_transcript(live["messages"]))
    print("\nstats:", agent_stats(live["messages"]))
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (wording varies) a `days_between` call, a `calculator` call with `74 * 45000`, and an answer with **3,330,000 COP**. If the model skipped a tool and answered from memory, rerun; if it keeps doing that, strengthen the system prompt ("never compute in your head"). That's the main tuning knob you have.

If LangSmith tracing is on, open the `m02-03-first-agent` run: you'll see `model` → `tools` → `model` → `tools` → `model` as nested steps with each request's tokens. This view is how you debug agents.

### What just happened?

- `create_agent` took a **model object** (with our timeouts and retries from `get_model`). It also accepts a string like `"groq:openai/gpt-oss-20b"` (via `init_chat_model`), but then you lose the settings `get_model` applies.
- **Don't pass a model with tools already bound** (`llm.bind_tools(...)`): `create_agent` binds the tools itself. The [v1 migration guide](https://docs.langchain.com/oss/python/migrate/langchain-v1) lists pre-bound models as unsupported; in our test with langchain 1.4.4 it didn't raise, which is worse: the behaviour is undefined, so a mistake goes unnoticed.
- The input and output are the **state**: a dict with `messages`. You add the user turn; the agent appends everything else.

---
## Part 2 · Every run needs a limit

The agent stops when the model answers without tool calls. A confused model may **never** do that. LangGraph counts *super-steps* (each node run is one) and raises `GraphRecursionError` at `recursion_limit`.

> **The default limit won't protect your bill.** The [docs](https://docs.langchain.com/oss/python/langgraph/graph-api#recursion-limit) say the default has been 1000 super-steps since LangGraph 1.0.6. In the installed `langgraph` 1.2.14 we measured more: a stuck scripted agent made **5000 model calls** before failing (`Recursion limit of 9999 reached`). With a real model, that's thousands of paid requests. **Always pass your own limit.**

One agent "turn" (model + tools) uses 2 super-steps, so `recursion_limit = 2 × max_tool_rounds + 1`.

In [ ]:
# 🧪 Offline: a model stuck in a loop, stopped by recursion_limit.
from langgraph.errors import GraphRecursionError

stuck = ScriptedChatModel(repeat_last=True, responses=[
    AIMessage("", tool_calls=[{"name": "calculator", "args": {"expression": "1 + 1"}, "id": "loop"}]),
])
stuck_agent = create_agent(stuck, tools=[calculator])

try:
    stuck_agent.invoke({"messages": [{"role": "user", "content": "Add one and one, forever."}]}, {"recursion_limit": 7})
except GraphRecursionError as exc:
    print("GraphRecursionError:", str(exc).splitlines()[0])
print("model calls before the stop:", len(stuck.calls))

**You should now see** `GraphRecursionError: Recursion limit of 7 reached ...` after **4** model calls (model, tools, model, tools, model, tools, model = 7 steps).

The error stops the run, but you get no answer and (without persistence) lose the work. The middleware version ends **gracefully** instead:

In [ ]:
# 🧪 Offline: ModelCallLimitMiddleware ends the run with a message instead of raising.
from langchain.agents.middleware import ModelCallLimitMiddleware

stuck.reset()
limited_agent = create_agent(stuck, tools=[calculator], middleware=[ModelCallLimitMiddleware(run_limit=3, exit_behavior="end")])
limited = limited_agent.invoke({"messages": [{"role": "user", "content": "Add one and one, forever."}]}, {"recursion_limit": 25})
print("model calls:", len(stuck.calls))
print("last message:", type(limited["messages"][-1]).__name__, "→", limited["messages"][-1].content)

**You should now see** 3 model calls and a final `AIMessage` saying the model call limit was reached. Use **both** in production: the middleware for a clean, user-facing stop; `recursion_limit` as the hard safety net. (`exit_behavior="error"` raises `ModelCallLimitExceededError` instead, from `langchain.agents.middleware.model_call_limit`.)

---
## Part 3 · System prompts, state, and dynamic prompts

`system_prompt` is a string (or a `SystemMessage`) that's fixed when you build the agent. Often the prompt depends on the request: the user's name, their currency, today's date. Two pieces make that possible:

- **`state_schema`**: extra fields in the agent's state, next to `messages`. It must be a `TypedDict` that subclasses `AgentState` (Pydantic and dataclass state schemas are not supported by `create_agent`).
- **`@dynamic_prompt`**: a middleware function that builds the system prompt from the request (state, runtime context) before **every** model call.

(Lesson 02.04 adds **runtime context**, the better place for read-only per-request data like a user id.)

In [ ]:
# 🧪 Offline: extra state + a prompt built per request.
from langchain.agents import AgentState
from langchain.agents.middleware import ModelRequest, dynamic_prompt


class FinanceState(AgentState):
    user_name: str
    home_currency: str


@dynamic_prompt
def personalized_prompt(request: ModelRequest) -> str:
    state = request.state
    return (f"{SYSTEM_PROMPT} The user is {state.get('user_name', 'unknown')}; "
            f"report money in {state.get('home_currency', 'COP')}.")


greeter = ScriptedChatModel(responses=["Hi Ana! Amounts will be in COP."])
personal_agent = create_agent(greeter, tools=[calculator], state_schema=FinanceState, middleware=[personalized_prompt])
out = personal_agent.invoke({"messages": [{"role": "user", "content": "Hi!"}], "user_name": "Ana", "home_currency": "COP"})

print("system prompt sent:", greeter.calls[0][0].content)
print("state keys back   :", sorted(out))

**You should now see** the prompt ending in *"The user is Ana; report money in COP."* and the state keys `home_currency`, `messages`, `user_name`. The extra fields travel with the state, so tools can read them too (02.04).

> `dynamic_prompt` replaces `system_prompt`'s role: use one or the other. Keep prompts **deterministic** for a given input; a prompt with the current time in seconds breaks provider prompt caching and makes runs hard to compare.

---
## Part 4 · Choosing the model per step

A common cost pattern: answer simple turns with the small model, switch to the big one when the conversation gets long or a tool failed. `@wrap_model_call` wraps each model call; `request.override(model=...)` swaps the model for that call only. Tools are bound to whichever model is chosen.

In [ ]:
# 🧪 Offline: small model for the first call, big model once tool results are in the history.
from langchain.agents.middleware import wrap_model_call
from langchain.messages import ToolMessage

small = ScriptedChatModel(model_name="small", responses=[
    AIMessage("", tool_calls=[{"name": "calculator", "args": {"expression": "1850000 * 12"}, "id": "s1"}])])
big = ScriptedChatModel(model_name="big", responses=["Your yearly rent is 22,200,000 COP."])


@wrap_model_call
def escalate_after_tools(request: ModelRequest, handler):
    uses_tools = any(isinstance(m, ToolMessage) for m in request.messages)
    return handler(request.override(model=big) if uses_tools else request)


router_agent = create_agent(small, tools=[calculator], middleware=[escalate_after_tools])
routed = router_agent.invoke({"messages": [{"role": "user", "content": "Rent is 1,850,000 a month. Yearly?"}]})
print("small model calls:", len(small.calls), "| big model calls:", len(big.calls))
print(routed["messages"][-1].content)

In [ ]:
# 💸 Live: 20b for tool selection, 120b once tool results arrive (3 short calls, one on the larger model).
if HAVE_GROQ_KEY:
    fast_llm = get_model("fast", temperature=0, reasoning_effort="low")
    reasoning_llm = get_model("reasoning", temperature=0, reasoning_effort="low")


    @wrap_model_call
    def live_escalate(request: ModelRequest, handler):
        uses_tools = any(isinstance(m, ToolMessage) for m in request.messages)
        return handler(request.override(model=reasoning_llm) if uses_tools else request)


    mixed = create_agent(fast_llm, tools=[calculator, days_between], system_prompt=SYSTEM_PROMPT, middleware=[live_escalate])
    mixed_out = mixed.invoke({"messages": [{"role": "user", "content": QUESTION}]}, {"recursion_limit": 12})
    for m in mixed_out["messages"]:
        if isinstance(m, AIMessage):
            print(f"{m.response_metadata.get('model_name', '?'):<22} {(m.tool_calls[0]['name'] if m.tool_calls else m.text)[:80]}")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** the first line from `openai/gpt-oss-20b` and later lines from `openai/gpt-oss-120b`. Check the price per million tokens in `config/models.yaml` before you make this the default: escalation pays off only when the big model saves retries or wrong answers.

---
## Part 5 · Migrating from `create_react_agent`

Most tutorials online (2024–2025) use `langgraph.prebuilt.create_react_agent`. It still imports, but it is deprecated: *"create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`."*

| Old (`create_react_agent`) | New (`create_agent`) |
|---|---|
| `from langgraph.prebuilt import create_react_agent` | `from langchain.agents import create_agent` |
| `prompt="..."` (str, message or callable) | `system_prompt="..."`; dynamic prompts via `@dynamic_prompt` |
| model node named `"agent"` | model node named `"model"` |
| `pre_model_hook` / `post_model_hook` | middleware (`before_model`, `after_model`) |
| dynamic model via a callable `model` | `@wrap_model_call` + `request.override(model=...)` |
| `state_schema` could be Pydantic | `TypedDict` subclass of `AgentState` only |
| `config={"configurable": {...}}` for per-run data | `context_schema` + `invoke(..., context=...)` (02.04) |
| pre-bound model allowed | not allowed: pass the plain model |
| `response_format` → extra LLM call at the end | `response_format` handled in the main loop (02.05) |

Full list: [LangChain v1 migration guide](https://docs.langchain.com/oss/python/migrate/langchain-v1).

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| Odd tool behaviour, tools you didn't pass | You passed `llm.bind_tools(...)` (unsupported, no error) | Pass the plain model; `create_agent` binds the tools |
| `GraphRecursionError` | Model kept calling tools | Fix the prompt/tool; set `recursion_limit` deliberately; add `ModelCallLimitMiddleware` |
| `KeyError: 'messages'` or the agent ignores your text | Input wasn't `{"messages": [...]}` | `agent.invoke({"messages": [{"role": "user", "content": "..."}]})` |
| Extra state key is silently missing | Not declared in `state_schema` | Subclass `AgentState` and pass `state_schema=` |
| `TypeError`/validation error with a Pydantic state | `create_agent` supports TypedDict state only | Use `class MyState(AgentState): ...` |
| Tutorial code uses `prompt=` or node `"agent"` | Pre-v1 `create_react_agent` code | Use the migration table above |
| The answer contains made-up numbers | Model skipped a tool | System prompt: "always use tools for ..."; check `agent_stats` in tests |

## ⚠️ Production notes

> ⚠️ **Production note: set limits on every agent call.**
> Pass `recursion_limit` on every `invoke`/`stream` (or bake it in with `agent.with_config({"recursion_limit": 12})`) and add `ModelCallLimitMiddleware(run_limit=...)` for a graceful stop. The framework default (1000 in the docs, more in the installed 1.2.14) is far above what any useful answer needs.

> ⚠️ **Production note: name your agents and runs.**
> `create_agent(..., name="savings_agent")` and `{"run_name": ..., "tags": [...]}` make traces searchable. In LangSmith, filter by tag to compare prompt versions; with no names, every trace is called `LangGraph`.

> ⚠️ **Production note: escalate models on evidence.**
> Model routing (`wrap_model_call`) multiplies the paths you must test. Start with one model; add routing only when evaluations (Module 07) show the small model failing on a measurable class of questions.

## 🏋️ Exercises

1. **(Easy)** Run the live agent with the question *"What day of the week is 2026-12-25?"*. Which tool does it use? (Hint: `get_today` and `add_days` are in `mastery.tools.DATE_TOOLS`; add them.)
2. **(Medium)** Give the stuck agent `ModelCallLimitMiddleware(run_limit=3, exit_behavior="error")`. Which exception do you get, and from which module?
3. **(Medium)** Add a `question_count: int` field to `FinanceState` and a `@dynamic_prompt` that says *"Be extra brief"* when `question_count > 3`. Pass it in the input and check the prompt in `greeter.calls`.
4. **(Stretch)** Write `budgeted(agent, max_tool_rounds)` that returns `agent.with_config({"recursion_limit": 2 * max_tool_rounds + 1})`. Test it with the stuck model: how many model calls happen for `max_tool_rounds=2`?

<details>
<summary><b>Show solutions</b></summary>

**1.** `add_days(start="2026-12-25", days=0)` (or `days_between`) returns a weekday; with `DATE_TOOLS` bound, the model usually calls `add_days`. It's a **Friday**.

**2.** `ModelCallLimitExceededError`, defined in `langchain.agents.middleware.model_call_limit` (it isn't re-exported from `langchain.agents.middleware`, so import it from the submodule).

**4.**
```python
def budgeted(agent, max_tool_rounds: int):
    return agent.with_config({"recursion_limit": 2 * max_tool_rounds + 1})

stuck.reset()
try:
    budgeted(stuck_agent, 2).invoke({"messages": [{"role": "user", "content": "loop"}]})
except GraphRecursionError:
    print(len(stuck.calls))   # 3: model, tools, model, tools, model
```
</details>

## ✅ Checkpoint

- [ ] I can build an agent with `create_agent(model, tools, system_prompt=...)` and invoke it with `{"messages": [...]}`.
- [ ] I can name the two graph nodes and the condition that ends the loop.
- [ ] I know the system prompt is sent every call but not stored in `messages`.
- [ ] I pass `recursion_limit` on every run and know what `ModelCallLimitMiddleware` adds.
- [ ] I can add state fields with `state_schema` and build prompts with `@dynamic_prompt`.
- [ ] I can swap models per call with `@wrap_model_call` and `request.override(model=...)`.
- [ ] I can translate a `create_react_agent` example into `create_agent`.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Open `site-packages/langgraph/_internal/_config.py` and find `DEFAULT_RECURSION_LIMIT`. Which environment variable overrides it? Compare its value with the 1000 the docs give, then search the [LangGraph releases](https://github.com/langchain-ai/langgraph/releases) for "recursion limit" to find when it changed. Log your finding in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- `create_agent` = the 02.02 loop as a graph (`model` ⇄ `tools`) plus error handling, streaming, middleware and tracing.
- The system prompt is configuration; `messages` is the history.
- Always bound a run: `recursion_limit` (hard stop) plus `ModelCallLimitMiddleware` (graceful stop).
- `state_schema` adds fields; `@dynamic_prompt` and `@wrap_model_call` customize each model call.
- `create_react_agent` code maps to `create_agent` with a few renames.

---
← [02-02-tool-calling-loop.ipynb](02-02-tool-calling-loop.ipynb) · [Module 02 README](README.md) · [02-04-tool-runtime-and-context.ipynb](02-04-tool-runtime-and-context.ipynb) →